In [ ]:
from datetime import datetime
from typing import List, Dict, Optional
import pandas as pd

# filename: gantt_contributions.py
# Author: GitHub Copilot
# Description:
#   Create a customizable Gantt chart with contribution flow for a project.
#   Contributors: "M. Yusuf Aykut, İbrahim Abu Shawish, Samet Ertuğrul Kurum, Samet Baturay"
#
# How to use:
#   - Run this file as a script, or copy functions into a Jupyter cell.
#   - Edit PROJECT_TASKS or pass your own tasks to build_gantt().
#   - The chart supports:
#       * Filtering by contributor
#       * Coloring by contributor or task group
#       * Interactive timeline zooming
#       * Optional contribution Sankey flow view
#
# Requirements:
#   pip install plotly pandas

import plotly.express as px
import plotly.graph_objects as go

# Default contributors (exact names as requested)
CONTRIBUTORS = [
    "M. Yusuf Aykut",
    "İbrahim Abu Shawish",
    "Samet Ertuğrul Kurum",
    "Samet Baturay",
]

# Example project tasks; customize as needed
PROJECT_TASKS: List[Dict] = [
    {
        "Task": "Requirements",
        "Group": "Planning",
        "Contributor": "M. Yusuf Aykut",
        "Start": "2025-01-02",
        "Finish": "2025-01-08",
        "Progress": 100,
    },
    {
        "Task": "Architecture",
        "Group": "Planning",
        "Contributor": ["M. Yusuf Aykut","İbrahim Abu Shawish"],
        "Start": "2025-01-05",
        "Finish": "2025-01-12",
        "Progress": 100,
    },
    {
        "Task": "Data Modeling",
        "Group": "Backend",
        "Contributor": "Samet Ertuğrul Kurum",
        "Start": "2025-01-10",
        "Finish": "2025-01-18",
        "Progress": 75,
    },
    {
        "Task": "API Development",
        "Group": "Backend",
        "Contributor": "M. Yusuf Aykut",
        "Start": "2025-01-15",
        "Finish": "2025-01-28",
        "Progress": 60,
    },
    {
        "Task": "Frontend UI",
        "Group": "Frontend",
        "Contributor": "Samet Baturay",
        "Start": "2025-01-20",
        "Finish": "2025-02-02",
        "Progress": 40,
    },
    {
        "Task": "Integration",
        "Group": "DevOps",
        "Contributor": "İbrahim Abu Shawish",
        "Start": "2025-01-25",
        "Finish": "2025-02-05",
        "Progress": 30,
    },
    {
        "Task": "Testing",
        "Group": "QA",
        "Contributor": "Samet Ertuğrul Kurum",
        "Start": "2025-02-01",
        "Finish": "2025-02-10",
        "Progress": 20,
    },
    {
        "Task": "Documentation",
        "Group": "QA",
        "Contributor": "Samet Baturay",
        "Start": "2025-02-03",
        "Finish": "2025-02-12",
        "Progress": 10,
    },
]

def _parse_dates(df: pd.DataFrame, start_col: str, end_col: str) -> pd.DataFrame:
    df = df.copy()
    df[start_col] = pd.to_datetime(df[start_col])
    df[end_col] = pd.to_datetime(df[end_col])
    return df

def _validate_tasks(tasks: List[Dict], contributors: List[str]) -> pd.DataFrame:
    df = pd.DataFrame(tasks)
    required_cols = {"Task", "Group", "Contributor", "Start", "Finish"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing task fields: {missing}")
    unknown_contribs = set(df["Contributor"]) - set(contributors)
    if unknown_contribs:
        raise ValueError(f"Unknown contributors in tasks: {unknown_contribs}")
    df = _parse_dates(df, "Start", "Finish")
    if (df["Finish"] < df["Start"]).any():
        bad = df[df["Finish"] < df["Start"]]
        raise ValueError(f"Finish before Start for tasks: {bad['Task'].tolist()}")
    # Duration and tooltip
    df["DurationDays"] = (df["Finish"] - df["Start"]).dt.days
    df["Progress"] = df.get("Progress", pd.Series([None]*len(df)))
    df["Hover"] = df.apply(
        lambda r: f"{r['Task']} | {r['Group']} | {r['Contributor']}\n"
                  f"{r['Start'].date()} → {r['Finish'].date()} ({r['DurationDays']}d)\n"
                  f"Progress: {r['Progress']}%", axis=1
    )
    return df

def build_gantt(
    tasks: List[Dict],
    contributors: Optional[List[str]] = None,
    color_by: str = "Contributor",  # "Contributor" or "Group"
    contributor_filter: Optional[List[str]] = None,
    show_progress_overlay: bool = True,
    title: str = "Project Gantt Chart with Contribution Flow",
) -> go.Figure:
    """
    Build a Plotly Gantt (timeline) chart.
    """
    contributors = contributors or CONTRIBUTORS
    df = _validate_tasks(tasks, contributors)

    # Filter by contributor if requested
    if contributor_filter:
        df = df[df["Contributor"].isin(contributor_filter)].copy()
        if df.empty:
            raise ValueError("No tasks after applying contributor_filter.")

    # Base Gantt timeline
    fig = px.timeline(
        df,
        x_start="Start",
        x_end="Finish",
        y="Task",
        color=color_by,
        hover_name="Task",
        hover_data={"Hover": True, "Start": True, "Finish": True, "DurationDays": True, "Progress": True, "Task": False},
    )
    fig.update_traces(hovertemplate="%{customdata[0]}")
    # Ensure tasks are ordered by start date
    df_sorted = df.sort_values(by=["Start", "Finish"])
    fig.update_yaxes(categoryorder="array", categoryarray=df_sorted["Task"].tolist())
    fig.update_layout(
        title=title,
        xaxis_title="Timeline",
        yaxis_title="Tasks",
        bargap=0.2,
        legend_title=color_by,
        hovermode="closest",
        template="plotly_white",
    )

    # Optional progress overlay: draw a semi-transparent bar within each task
    if show_progress_overlay and "Progress" in df.columns and df["Progress"].notnull().any():
        overlay_traces = []
        for _, r in df.iterrows():
            if pd.isna(r["Progress"]):
                continue
            progress_ratio = max(0.0, min(1.0, float(r["Progress"]) / 100.0))
            progress_end = r["Start"] + (r["Finish"] - r["Start"]) * progress_ratio
            overlay_traces.append(go.Bar(
                x=[(progress_end - r["Start"]).days],
                y=[r["Task"]],
                base=r["Start"],
                orientation="h",
                marker=dict(color="rgba(0,0,0,0.15)"),
                showlegend=False,
                hoverinfo="skip",
            ))
        for t in overlay_traces:
            fig.add_trace(t)

    # Add range slider and buttons
    fig.update_layout(
        xaxis=dict(
            rangeselector=dict(
                buttons=list([
                    dict(count=7, label="1w", step="day", stepmode="backward"),
                    dict(count=14, label="2w", step="day", stepmode="backward"),
                    dict(count=1, label="1m", step="month", stepmode="backward"),
                    dict(step="all")
                ])
            ),
            rangeslider=dict(visible=True),
            type="date"
        )
    )
    return fig

def build_contribution_sankey(tasks: List[Dict], contributors: Optional[List[str]] = None, title: str = "Contribution Flow") -> go.Figure:
    """
    Build a Sankey diagram showing flow from contributors to task groups.
    """
    contributors = contributors or CONTRIBUTORS
    df = _validate_tasks(tasks, contributors)

    # Nodes: contributors + groups
    groups = sorted(df["Group"].unique().tolist())
    nodes = contributors + groups
    node_index = {name: i for i, name in enumerate(nodes)}

    # Aggregate contributions by duration (or count)
    agg = df.groupby(["Contributor", "Group"])["DurationDays"].sum().reset_index()

    links = dict(
        source=[node_index[c] for c in agg["Contributor"]],
        target=[node_index[g] for g in agg["Group"]],
        value=agg["DurationDays"].tolist(),
        label=[f"{c} → {g}: {v}d" for c, g, v in agg.values]
    )

    fig = go.Figure(data=[go.Sankey(
        node=dict(
            pad=15,
            thickness=20,
            line=dict(color="black", width=0.5),
            label=nodes,
            color=["#4C78A8"] * len(contributors) + ["#F58518"] * len(groups),
        ),
        link=links
    )])
    fig.update_layout(title_text=title, font_size=12, template="plotly_white")
    return fig

def main():
    # Build charts
    gantt = build_gantt(
        PROJECT_TASKS,
        contributors=CONTRIBUTORS,
        color_by="Contributor",  # or "Group"
        contributor_filter=None,  # e.g., ["M. Yusuf Aykut", "Samet Baturay"]
        show_progress_overlay=True,
        title="Project Gantt Chart (Contributors: M. Yusuf Aykut, İbrahim Abu Shawish, Samet Ertuğrul Kurum, Samet Baturay)"
    )
    sankey = build_contribution_sankey(PROJECT_TASKS, contributors=CONTRIBUTORS, title="Contribution Flow by Duration (days)")

    # Display in interactive window (for scripts). In Jupyter, just display the figures.
    gantt.show()
    sankey.show()

if __name__ == "__main__":
    main()